In [92]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import re
import joblib
import string

In [94]:
fake = pd.read_csv('Fake.csv')
true = pd.read_csv('True.csv')


In [95]:
fake.head() 

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [98]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [100]:
fake['class']=0
true['class']=1

In [102]:
data = pd.concat([fake, true], axis = 0)

In [104]:
data.sample(10)

,title,text,subject,date,class
12304,Zimbabwe's Mnangagwa calls for end to Western ...,HARARE (Reuters) - New President Emmerson Mnan...,worldnews,"December 14, 2017",1
16282,U.S. backs Spanish efforts to block break-away...,WASHINGTON (Reuters) - The U.S. State Departme...,worldnews,"October 27, 2017",1
18881,'Red Scare' puts pressure on Indonesian president,JAKARTA (Reuters) - Indonesian police will dep...,worldnews,"September 27, 2017",1
9980,"Top Democrat, in letter, blasts Valeant CEO fo...",WASHINGTON (Reuters) - A top Democrat in the U...,politicsNews,"April 12, 2016",1
15147,Ground Zero Mosque Was NOT Defeated: Three Sto...,Will the new 3-story Islamic Museum include pi...,politics,"Sep 29, 2015",0
20817,Togo opposition calls for president to quit as...,LOME (Reuters) - Togo s opposition chief calle...,worldnews,"September 6, 2017",1
20331,MATT DAMON Says America Needs IMMEDIATE GUN BA...,It warms the heart to know people as important...,left-news,"Jul 6, 2016",0
12051,Paraguay senator with dictatorship ties to run...,ASUNCION (Reuters) - Paraguayan Senator Mario ...,worldnews,"December 18, 2017",1
20485,South Africa's Ramaphosa steps up criticism ah...,JOHANNESBURG (Reuters) - South African Deputy ...,worldnews,"September 10, 2017",1
9510,USA! USA! USA! Troops Cheer President Trump Du...,President Trump received a warm welcome when h...,politics,"Nov 5, 2017",0


In [106]:
data = data.drop(['title', 'subject', 'date'], axis=1)

In [108]:
data.sample(10)

,text,class
5477,Anyone with a brain knows that Donald Trump is...,0
11116,WASHINGTON (Reuters) - The U.S. Supreme Court ...,1
7214,BERLIN (Reuters) - President Barack Obama said...,1
19775,Here is the list of Republicans who won t supp...,0
7879,Two pasty white Republican men decided that ou...,0
1232,Rep. Maxine Waters is taking a page right out ...,0
13190,"If you believe this excuse, you ll believe any...",0
16690,Is this not the craziest thing ever? No wonder...,0
9099,WASHINGTON/NEW YORK (Reuters) - Presumptive De...,1
20688,HOUSTON (Reuters) - Monster Hurricane Irma has...,1


In [110]:
data.reset_index(inplace = True)

In [112]:
data

,index,text,class
0,0,Donald Trump just couldn t wish all Americans ...,0
1,1,House Intelligence Committee Chairman Devin Nu...,0
2,2,"On Friday, it was revealed that former Milwauk...",0
3,3,"On Christmas day, Donald Trump announced that ...",0
4,4,Pope Francis used his annual Christmas Day mes...,0
...,...,...,...
44893,21412,BRUSSELS (Reuters) - NATO allies on Tuesday we...,1
44894,21413,"LONDON (Reuters) - LexisNexis, a provider of l...",1
44895,21414,MINSK (Reuters) - In the shadow of disused Sov...,1
44896,21415,MOSCOW (Reuters) - Vatican Secretary of State ...,1


In [114]:
data.drop(['index'], axis =1, inplace =True)

In [116]:
data

,text,class
0,Donald Trump just couldn t wish all Americans ...,0
1,House Intelligence Committee Chairman Devin Nu...,0
2,"On Friday, it was revealed that former Milwauk...",0
3,"On Christmas day, Donald Trump announced that ...",0
4,Pope Francis used his annual Christmas Day mes...,0
...,...,...
44893,BRUSSELS (Reuters) - NATO allies on Tuesday we...,1
44894,"LONDON (Reuters) - LexisNexis, a provider of l...",1
44895,MINSK (Reuters) - In the shadow of disused Sov...,1
44896,MOSCOW (Reuters) - Vatican Secretary of State ...,1


In [118]:
def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r'\[.*?\]', '', text)  # remove text inside square brackets
    text = re.sub(r'\W', ' ', text)  # remove all non-word characters (keep only letters, numbers, _)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # remove all URLs
    text = re.sub(r'<.*?>+', '', text)  # remove HTML tags
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)  # remove punctuation
    text = re.sub(r'\n', ' ', text)  # remove newline chars
    text = re.sub(r'\w*\d\w*', '', text)  # remove words containing digits
    return text.strip()

In [120]:
data['text'] = data['text'].apply(clean_text)

In [121]:
X=data['text']
y=data['class']

X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.25,random_state=42)

In [125]:
# converting text to numeric data

In [127]:
vectorizer = TfidfVectorizer()
xv_train = vectorizer.fit_transform(X_train)
xv_test = vectorizer.transform(X_test)


In [128]:
lr = LogisticRegression()
lr.fit(xv_train, y_train)

LogisticRegression()

In [131]:
prediction= lr.predict(xv_test)
lr.score(xv_test, y_test)

0.9858351893095768

In [133]:
print(classification_report(y_test, prediction))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5895
           1       0.98      0.99      0.99      5330

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [135]:
joblib.dump(vectorizer, 'vectorizer.jb')
joblib.dump(lr,'lr_model.jb')

['lr_model.jb']